# Revenue & Churn Intelligence — Rule-Based RFM Scoring

Run in Google Colab, connected to BigQuery.

Deliberately rule-based (no classifier/model training) — meant to read as Data Analyst work, not Data Scientist work.

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='revenue-churn-analysis')

query = '''
SELECT *
FROM `revenue-churn-analysis.ga4_analysis.user_purchase_summary`
'''
df = client.query(query).to_dataframe()
print(df.shape)
df.head()

## Score each customer 1-5 on Recency, Frequency, Monetary

In [ ]:
# Recency: LOWER days_since_last_purchase = better, so labels are reversed (5 = most recently active)
df['R_score'] = pd.qcut(df['days_since_last_purchase'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency: higher total_transactions = better. Many customers tie at exactly 1 transaction,
# so .rank(method='first') breaks ties for qcut to work cleanly.
df['F_score'] = pd.qcut(df['total_transactions'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Monetary: higher total_revenue = better, straightforward quintiles
df['M_score'] = pd.qcut(df['total_revenue'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

df['RFM_score'] = df['R_score'].astype(str) + df['F_score'].astype(str) + df['M_score'].astype(str)

## Translate scores into segment labels

Rule order matters — checked top to bottom, first match wins. "At Risk - High Value" is deliberately checked before plain "At Risk".

**Debugging note:** the first version left a gap (r>=4, f>=4, m=3) that dumped ~1,000 customers into a lazy "Other" bucket — the biggest segment. Adding the explicit "Loyal Low-Spender" rule closed the gap; "Other" dropped to exactly 0.

In [ ]:
def rfm_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 4 and f >= 4 and m < 4:
        return 'Loyal Low-Spender'
    elif r >= 4 and f == 3:
        return 'Promising'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'At Risk - High Value'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost'
    elif r == 3:
        return 'Needs Attention'
    else:
        return 'Other'  # should be unreachable — verified 0 rows land here

df['segment'] = df.apply(rfm_segment, axis=1)
df['segment'].value_counts()

## Push the scored table back to BigQuery

In [ ]:
table_id = 'revenue-churn-analysis.ga4_analysis.rfm_segments'
job = client.load_table_from_dataframe(df, table_id)
job.result()
print(f"Loaded {job.output_rows} rows into {table_id}")